# ANOVA Analysis

Phân tích phương sai (ANOVA) để so sánh mean của các biến số giữa các nhóm category.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import f_oneway, kruskal
import statsmodels.api as sm
from statsmodels.formula.api import ols
from functools import reduce

In [8]:
# Load cleaned data
df = pd.read_csv('../data/youtube_no_outliers_clean_final.csv')

numerical_cols = ['duration', 'bitrate', 'bitrate(video)', 'height', 'width',
                  'frame rate', 'frame rate(est.)', 'views', 'likes', 'comments']

In [9]:
df['codec'].value_counts()

codec
h264     17528
vp8         47
flv1         9
mpeg4        2
Name: count, dtype: int64

## 1. One-Way ANOVA Test

Kiểm tra xem có sự khác biệt về mean giữa các category hoặc codec hay không.

In [10]:
# Áp dụng biến đổi log(1+x)
df['log_views'] = np.log1p(df['views'])
df['log_likes'] = np.log1p(df['likes'])

# Biến phân loại và biến phụ thuộc
categorical_features = ['codec', 'category']
dependent_variables = ['log_views', 'log_likes']
alpha = 0.05

# --- 1. ANOVA 1-WAY (Tác động riêng lẻ) ---
anova_results_1way = []

for dep_var in dependent_variables:
    for category in categorical_features:
        # Group data for f_oneway test
        grouped_data = df.groupby(category)[dep_var].apply(list)
        f_value, p_value = stats.f_oneway(*grouped_data)

        anova_results_1way.append({
            'Dependent_Variable': dep_var,
            'Factor': category,
            'F-value': f_value,
            'P-Value': p_value
        })

df_anova_1way = pd.DataFrame(anova_results_1way)

print(df_anova_1way)

  Dependent_Variable    Factor    F-value        P-Value
0          log_views     codec   1.668727   1.713868e-01
1          log_views  category  80.945412  4.910192e-241
2          log_likes     codec   1.731225   1.581944e-01
3          log_likes  category  74.449502  2.053191e-221


## 2. Two-Way ANOVA Test

Tác động kết hợp và Tương tác

In [11]:
anova_results_2way = []
factors_formula = 'C(codec) * C(category)'

for dep_var in dependent_variables:
    formula = f'{dep_var} ~ {factors_formula}'
    model = ols(formula=formula, data=df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    for index, row in anova_table.iterrows():
        if index != 'Residual':
            anova_results_2way.append({
                'Dependent_Variable': dep_var,
                'Factor': index,
                'F-value': row['F'],
                'P-Value': row['PR(>F)']
            })

df_anova_2way = pd.DataFrame(anova_results_2way)

print(df_anova_2way)

print(f"\nCÁC YẾU TỐ CÓ Ý NGHĨA THỐNG KÊ (P-Value < {alpha}) TRONG ANOVA 2-WAY:")
print(df_anova_2way[df_anova_2way['P-Value'] < alpha])

  Dependent_Variable                Factor       F-value       P-Value
0          log_views              C(codec)  3.915077e+01  1.083561e-17
1          log_views           C(category)  4.330073e-09  1.000000e+00
2          log_views  C(codec):C(category)  1.545034e-01  9.999735e-01
3          log_likes              C(codec)  5.345271e+01  7.181258e-24
4          log_likes           C(category)  9.323388e-08  1.000000e+00
5          log_likes  C(codec):C(category)  7.413066e-02  9.999999e-01

CÁC YẾU TỐ CÓ Ý NGHĨA THỐNG KÊ (P-Value < 0.05) TRONG ANOVA 2-WAY:
  Dependent_Variable    Factor    F-value       P-Value
0          log_views  C(codec)  39.150773  1.083561e-17
3          log_likes  C(codec)  53.452712  7.181258e-24


/Users/dangvanvy/PycharmProjects/PTDLTQ-IE313/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '
/Users/dangvanvy/PycharmProjects/PTDLTQ-IE313/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 15, but rank is 9
  warnings.warn('covariance of constraints does not have full '
/Users/dangvanvy/PycharmProjects/PTDLTQ-IE313/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 45, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/Users/dangvanvy/PycharmProjects/PTDLTQ-IE313/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covari

* **Category (Danh mục)**: Rõ ràng là yếu tố mạnh nhất khi đứng độc lập (ANOVA 1-Way).Codec (Định dạng): Có tác động đáng kể khi nằm trong mô hình 2-Way, nhưng không có tác động đáng kể khi đứng độc lập (ANOVA 1-Way).
* **Vấn đề về Mô hình 2-Way**: Việc $P$-Value của category bằng $1.000$ trong mô hình 2-Way gần như chắc chắn là không chính xác do các cảnh báo về covariance of constraints does not have full rank. Mô hình 2-Way đang bị lỗi và không thể ước tính tác động của category một cách đáng tin cậy.

In [12]:
df.groupby(['codec', 'category']).size()

codec  category            
flv1   Comedy                     1
       Entertainment              1
       Music                      1
       Pets & Animals             2
       Sports                     2
       Travel & Events            2
h264   Autos & Vehicles         798
       Comedy                  1169
       Education                643
       Entertainment           2243
       Film & Animation         586
       Gaming                  1418
       Howto & Style            340
       Music                   2955
       News & Politics          636
       Nonprofits & Activis     225
       People & Blogs          3935
       Pets & Animals           472
       Science & Technology     282
       Shows                     18
       Sports                  1223
       Travel & Events          585
mpeg4  Entertainment              1
       People & Blogs             1
vp8    Comedy                     6
       Education                  1
       Entertainment              5
